In [1]:
# Imports, device, and seed

import os, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, BatchNorm
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from scipy.sparse import issparse
from pathlib import Path
from joblib import dump, load

# Reproducibility
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [2]:
# Utility helpers
def make_undirected(edge_index_np):
    rows, cols = edge_index_np
    mask = rows != cols
    rows, cols = rows[mask], cols[mask]
    rows2 = np.concatenate([rows, cols])
    cols2 = np.concatenate([cols, rows])
    edges = np.vstack([rows2, cols2]).T
    edges = np.unique(edges, axis=0)
    return edges.T

def split_masks(n, train=0.7, val=0.15, test=0.15, seed=42):
    idx = np.random.RandomState(seed).permutation(n)
    n_train = int(train*n); n_val = int(val*n)
    train_idx = idx[:n_train]
    val_idx   = idx[n_train:n_train+n_val]
    test_idx  = idx[n_train+n_val:]
    masks = {}
    for name, idc in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
        m = torch.zeros(n, dtype=torch.bool)
        m[idc] = True
        masks[name] = m
    return masks

In [3]:
# Load dataset

CSV_PATH = "Concrete_Data.csv"          
TARGET_COL = "Concrete compressive strength(MPa, megapascals) "      

df = pd.read_csv(CSV_PATH)
assert TARGET_COL in df.columns, f"Target missing; check column names: {df.columns.tolist()}"
print("Dataset:", df.shape)

Dataset: (1030, 9)


In [4]:
def make_X(df, mode="A"):
    if mode == "A":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Blast Furnace Slag (component 2)(kg in a m^3 mixture)",
            "Fly Ash (component 3)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Coarse Aggregate  (component 6)(kg in a m^3 mixture)",
            "Fine Aggregate (component 7)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "B":
        cols = [
            "Cement (component 1)(kg in a m^3 mixture)",
            "Water  (component 4)(kg in a m^3 mixture)",
            "Superplasticizer (component 5)(kg in a m^3 mixture)",
            "Age (day)"
        ]
        return df[cols].copy()

    if mode == "C":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
        tmp = df[[c_cem,c_slag,c_fly,c_wat,c_sp,c_ca,c_fa,c_age]].copy()
        tmp["Binder"]    = tmp[c_cem] + tmp[c_slag] + tmp[c_fly]
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["Fluid"]     = tmp[c_wat] + tmp[c_sp]
        eps = 1e-9
        tmp["w_b"]   = tmp[c_wat] / (tmp["Binder"] + eps)
        tmp["agg_b"] = tmp["Aggregate"] / (tmp["Binder"] + eps)
        tmp["sp_b"]  = tmp[c_sp] / (tmp["Binder"] + eps)
        return tmp[["Binder","Aggregate","Fluid","w_b","agg_b","sp_b", c_age]].copy()

    if mode == "D":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"

        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
    
        return tmp[[c_cem, c_slag, c_fly, c_wat, c_sp, "Aggregate", c_age]].copy()


    if mode == "E":
        c_cem  = "Cement (component 1)(kg in a m^3 mixture)"
        c_slag = "Blast Furnace Slag (component 2)(kg in a m^3 mixture)"
        c_fly  = "Fly Ash (component 3)(kg in a m^3 mixture)"
        c_wat  = "Water  (component 4)(kg in a m^3 mixture)"
        c_sp   = "Superplasticizer (component 5)(kg in a m^3 mixture)"
        c_ca   = "Coarse Aggregate  (component 6)(kg in a m^3 mixture)"
        c_fa   = "Fine Aggregate (component 7)(kg in a m^3 mixture)"
        c_age  = "Age (day)"
    
        tmp = df[[c_cem, c_slag, c_fly, c_wat, c_sp, c_ca, c_fa, c_age]].copy()
        tmp["Aggregate"] = tmp[c_ca] + tmp[c_fa]
        tmp["SEM"] = tmp[c_slag] + tmp[c_fly]
    
        return tmp[[c_cem, c_wat, c_sp, "Aggregate", "SEM", c_age]].copy()


In [5]:
# Split features/target

y_raw = df[TARGET_COL].to_numpy().reshape(-1, 1)
X_df = make_X(df, mode="D")   
print("Using columns:", list(X_df.columns))

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

def build_ohe():
    try:

        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", build_ohe())
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ],
    remainder="drop",
    sparse_threshold=0.3  
)

X = preprocess.fit_transform(X_df)
if issparse(X):
    X = X.toarray()


from sklearn.preprocessing import StandardScaler as _SS
y_scaler = _SS().fit(y_raw)
y = y_scaler.transform(y_raw).ravel()

print("X shape:", X.shape, "| y shape:", y.shape)
print("Numeric cols:", len(num_cols), "| Categorical cols:", len(cat_cols))

Using columns: ['Cement (component 1)(kg in a m^3 mixture)', 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)', 'Fly Ash (component 3)(kg in a m^3 mixture)', 'Water  (component 4)(kg in a m^3 mixture)', 'Superplasticizer (component 5)(kg in a m^3 mixture)', 'Aggregate', 'Age (day)']
X shape: (1030, 7) | y shape: (1030,)
Numeric cols: 7 | Categorical cols: 0


In [6]:
# Build a kNN graph in feature space (undirected)

k = 5 
nbrs = NearestNeighbors(n_neighbors=min(k+1, X.shape[0]), metric="euclidean")
nbrs.fit(X)
A = nbrs.kneighbors_graph(X, n_neighbors=min(k+1, X.shape[0]), mode="connectivity")
A = A.maximum(A.T)

rows, cols = A.nonzero()
edge_index_np = np.vstack([rows, cols])
edge_index_np = make_undirected(edge_index_np)

edge_index = torch.tensor(edge_index_np, dtype=torch.long)
x = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

data = Data(x=x, edge_index=edge_index, y=y_t)
data

Data(x=[1030, 7], edge_index=[2, 6616], y=[1030])

In [7]:
#Train/Val/Test node masks

masks = split_masks(data.num_nodes, train=0.70, val=0.15, test=0.15, seed=42)
data.train_mask = masks["train"]
data.val_mask = masks["val"]
data.test_mask = masks["test"]
{ k: int(v.sum().item()) for k, v in masks.items() }

{'train': 721, 'val': 154, 'test': 155}

In [8]:
# Define a baseline GNN (GraphSAGE for regression)

class SAGENet(nn.Module):
    def __init__(self, in_channels, hidden=128, layers=3, dropout=0.25, aggr="mean"):
        super().__init__()
        assert aggr in {"mean", "max", "add"}, "aggr must be 'mean', 'max', or 'add'"

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(SAGEConv(in_channels, hidden, aggr=aggr))
        self.bns.append(BatchNorm(hidden))

        for _ in range(layers - 2):
            self.convs.append(SAGEConv(hidden, hidden, aggr=aggr))
            self.bns.append(BatchNorm(hidden))

        self.out_lin = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out_lin(x).view(-1)


model = SAGENet(in_channels=data.num_features, hidden=128, layers=3, dropout=0.25, aggr="mean").to(device)
data = data.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

In [9]:
# Train with early stopping

CKPT_DIR = Path.cwd() / "artifacts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path = CKPT_DIR / "gnn_best.pt"

best_val = float("inf")
patience, bad = 30, 0
history = []

for epoch in range(1, 1001):
    model.train()
    optimizer.zero_grad()

    pred = model(data.x, data.edge_index)
    loss = criterion(pred[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    # Validation only during training
    model.eval()
    with torch.no_grad():
        pred_all = model(data.x, data.edge_index)
        val_loss = criterion(
            pred_all[data.val_mask],
            data.y[data.val_mask]
        ).item()

    # Store only train and validation loss
    history.append(
        (epoch, float(loss.item()), val_loss)
    )

    # Early stopping based only on validation loss
    if val_loss < best_val - 1e-4:
        best_val = val_loss
        bad = 0

        try:
            tmp_path = CKPT_DIR / f"tmp_epoch{epoch}.pt"
            torch.save(model.state_dict(), tmp_path.open("wb"))
            tmp_path.replace(best_path)

        except Exception as e:
            fb = CKPT_DIR / f"gnn_best_epoch{epoch}.pt"
            torch.save(model.state_dict(), fb.open("wb"))
            print(
                f"[WARN] Could not write {best_path.name}: "
                f"{e}. Saved fallback: {fb.name}"
            )
    else:
        bad += 1

    if epoch % 25 == 0 or bad == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"train {loss.item():.4f} | "
            f"val {val_loss:.4f}"
        )

    if bad >= patience:
        print(
            f"Early stop at epoch {epoch}. "
            f"Best val {best_val:.4f}"
        )
        break

# Load best validation-selected model
if best_path.exists():
    model.load_state_dict(
        torch.load(best_path, map_location=device)
    )
    print(f"Loaded best checkpoint from: {best_path}")
else:
    print("[WARN] No best checkpoint found; using last-epoch weights.")

Epoch    1 | train 1.1596 | val 0.7840
Epoch    2 | train 0.8647 | val 0.6545
Epoch    3 | train 0.6862 | val 0.5558
Epoch    4 | train 0.5977 | val 0.4893
Epoch    5 | train 0.6046 | val 0.4431
Epoch    6 | train 0.6002 | val 0.4141
Epoch    7 | train 0.5128 | val 0.3952
Epoch    8 | train 0.5025 | val 0.3849
Epoch    9 | train 0.4669 | val 0.3805
Epoch   16 | train 0.3631 | val 0.3691
Epoch   17 | train 0.3563 | val 0.3493
Epoch   18 | train 0.3573 | val 0.3282
Epoch   19 | train 0.3500 | val 0.3055
Epoch   20 | train 0.3152 | val 0.2848
Epoch   21 | train 0.3068 | val 0.2668
Epoch   22 | train 0.3306 | val 0.2519
Epoch   23 | train 0.3181 | val 0.2402
Epoch   24 | train 0.3233 | val 0.2314
Epoch   25 | train 0.3020 | val 0.2247
Epoch   26 | train 0.3034 | val 0.2191
Epoch   27 | train 0.2828 | val 0.2149
Epoch   28 | train 0.2920 | val 0.2134
Epoch   41 | train 0.2757 | val 0.2100
Epoch   42 | train 0.2629 | val 0.2045
Epoch   43 | train 0.2470 | val 0.1998
Epoch   44 | train 0.2529

In [10]:
# Evaluate in original units

import torch

@torch.no_grad()
def evaluate_split(mask, name):
    model.eval()
    pred_scaled = model(data.x, data.edge_index).detach().cpu().numpy().reshape(-1, 1)
    y_scaled    = data.y.detach().cpu().numpy().reshape(-1, 1)

    # invert target scaling
    y_true = y_scaler.inverse_transform(y_scaled).ravel()
    y_pred = y_scaler.inverse_transform(pred_scaled).ravel()
    m = mask.cpu().numpy()

    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    rmse = root_mean_squared_error(y_true[m], y_pred[m])
    mae  = mean_absolute_error(y_true[m], y_pred[m])
    r2   = r2_score(y_true[m], y_pred[m])
    mape = np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m])) * 100
    print(f"{name:5s} | RMSE {rmse:.4f} | MAE {mae:.4f} | R² {r2:.4f} | MAPE {mape:.2f}%")
    return rmse, mae, r2, mape

evaluate_split(data.train_mask, "Train")
evaluate_split(data.val_mask,   "Val")
evaluate_split(data.test_mask,  "Test")

Train | RMSE 4.6383 | MAE 3.5035 | R² 0.9267 | MAPE 12.18%
Val   | RMSE 5.2868 | MAE 4.1172 | R² 0.8848 | MAPE 14.55%
Test  | RMSE 5.4157 | MAE 4.2243 | R² 0.8806 | MAPE 15.29%


(5.415682, 4.224348, 0.8805736303329468, 15.285356342792511)

In [11]:
import sys
import torch
import torch_geometric
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
PyTorch Geometric: 2.8.0.post1
scikit-learn: 1.5.1
